# 🌿 Crop Disease Detection Pipeline

This notebook runs the MobileNetV2 transfer learning pipeline for crop disease classification. 
Since it's running on Google Colab, it bypasses any potential local CPU/hardware errors on your system!

### Step 1: Run the cell below to verify TensorFlow works in Colab
*(Click the code cell below and press **Shift + Enter** or the **Play button** on the left)*

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.callbacks import ModelCheckpoint

def build_transfer_learning_model(num_classes=38):
    """Builds a MobileNetV2 transfer learning model."""
    input_shape = (224, 224, 3)
    
    # Load pre-trained base model
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=input_shape)
    base_model.trainable = False  # Freeze base model
    
    # Custom Head
    inputs = Input(shape=input_shape)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    x = base_model(x, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.2)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=0.0001), loss=CategoricalCrossentropy(), metrics=['accuracy'])
    
    return model, base_model

def fine_tune_model(model, base_model):
    """Unfreezes the top 20 layers for fine-tuning."""
    print("\n--- Preparing Fine-Tuning Setup ---")
    base_model.trainable = True
    
    layers_to_unfreeze = 20
    freeze_until = len(base_model.layers) - layers_to_unfreeze
    
    for layer in base_model.layers[:freeze_until]:
        layer.trainable = False
    for layer in base_model.layers[freeze_until:]:
        layer.trainable = True
        
    # Recompile with very tiny learning rate
    model.compile(optimizer=Adam(learning_rate=1e-5), loss=CategoricalCrossentropy(), metrics=['accuracy'])
    return model

# ==========================================
# Execute the setup to test the environment
# ==========================================
NUM_CLASSES = 38

print("1. Building Feature Extraction Model...")
model, base_model = build_transfer_learning_model(num_classes=NUM_CLASSES)
print("✅ Base Model Built Successfully!")

print("\n2. Preparing the Fine-Tuning architecture...")
model = fine_tune_model(model, base_model)
print("✅ Fine-Tuning Architecture Setup Successfully!")

print("\n🚀 TensorFlow is fully working in this Colab session!")
model.summary()
